In [1]:
from scipy.optimize import linprog
import numpy as np
import pandas as pd

In [2]:
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
import pandas as pd

# パラメータの設定
B = 100  # 総予算
w = [0.1, 0.1, 0.1, 4, 4]  # 投資項目の重み
alpha = [0.1, 0.1, 0.1, 0.3, 0.2]  # 各投資項目の最低割合
beta = [0.2, 0.2, 0.2, 0.6, 0.6]  # 各投資項目の最大割合

# 変数の定義
x = [[LpVariable(f"x_{i}_{g}", lowBound=0) for g in range(5)] for i in range(5)]

# 問題の定義
model = LpProblem(name="budget-allocation", sense=LpMaximize)

# 目的関数
model += lpSum(w[i] * x[i][g] for i in range(5) for g in range(5))

# 制約条件
# 総予算
model += lpSum(x[i][g] for i in range(5) for g in range(5)) <= B

# 各投資項目の割合
for i in range(5):
    model += lpSum(x[i][g] for g in range(5)) >= alpha[i] * B
    model += lpSum(x[i][g] for g in range(5)) <= beta[i] * B

# 最適化
model.solve()

slt1 = []
slt2 = []
# 結果の表示
for i in range(5):
    for g in range(5):
        print(f"x_{i}_{g} = {x[i][g].value()}")
        slt1.append(f'x[{i}][{g}]')
        slt2.append(x[i][g].value())
solution1 = pd.DataFrame(slt1)
solution2 = pd.DataFrame(slt2)
solution = pd.concat([solution1, solution2], axis=1)
solution.to_csv(r'/Users/kurozuhajime/Desktop/test.csv')

print(f"Objective value: {model.objective.value()}")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/5v/b1rgw4cx4dsdgq25qzpwcwk40000gn/T/225fff543ec846a6b07389275a19cfa6-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/5v/b1rgw4cx4dsdgq25qzpwcwk40000gn/T/225fff543ec846a6b07389275a19cfa6-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 16 COLUMNS
At line 117 RHS
At line 129 BOUNDS
At line 130 ENDATA
Problem MODEL has 11 rows, 25 columns and 75 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 1 (-10) rows, 5 (-20) columns and 5 (-70) elements
0  Obj 203 Dual inf 8.299995 (5)
2  Obj 283
Optimal - objective value 283
After Postsolve, objective 283, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 283 - 2 iterations time 0.002, Presolve 0.00
Option for printingOptions changed from norm

In [3]:
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus

# パラメータの定義
num_investments = 5  # 投資項目数（健康、教育、経済、CO2削減、マテリアル削減）
num_generations = 5  # 世代数（赤ちゃん、こども、若者、壮年、高齢者）
total_budget = 100  # 総予算

# 各投資項目の重み（重要度: 高いほど優先順位が高い）
weights = [0.2, 0.5, 0.1, 8, 11]  # 健康, 教育, 経済, CO2削減, マテリアル削減

# 投資項目ごとの予算割合の下限と上限
min_ratios = [0.1, 0.15, 0.1, 0.2, 0.05]  # 最小割合
max_ratios = [0.3, 0.4, 0.2, 0.8, 0.8]  # 最大割合

# 制約のための変数変換
min_budget = [min_ratios[i] * total_budget for i in range(num_investments)]
max_budget = [max_ratios[i] * total_budget for i in range(num_investments)]

# モデルの初期化
model = LpProblem(name="budget_allocation", sense=LpMaximize)

# 変数の定義（各投資項目 i に対する世代 g への割り当て額）
x = [[LpVariable(f"x_{i}_{g}", lowBound=0) for g in range(num_generations)] for i in range(num_investments)]

# 目的関数の定義（アウトカムの最大化）
model += lpSum(weights[i] * x[i][g] for i in range(num_investments) for g in range(num_generations)), "Total Outcome"

# 制約条件 1: 総予算制約
model += lpSum(x[i][g] for i in range(num_investments) for g in range(num_generations)) <= total_budget, "Total Budget"

# 制約条件 2: 各投資項目の割り当て制約（下限と上限）
for i in range(num_investments):
    model += lpSum(x[i][g] for g in range(num_generations)) >= min_budget[i], f"Min_Budget_Investment_{i}"
    model += lpSum(x[i][g] for g in range(num_generations)) <= max_budget[i], f"Max_Budget_Investment_{i}"

# 制約条件 3: 非負制約（各割り当て額は0以上）
# PuLPでは変数定義時に lowBound=0 を指定しているため、非負制約は自動的に保証されます。

# 最適化実行
model.solve()

# 結果の出力
print(f"Status: {model.status} ({LpStatus[model.status]})")
print(f"Objective Value (Total Outcome): {model.objective.value()}\n")

slt1 = []
slt2 = []
for i in range(num_investments):
    for g in range(num_generations):
        print(f"Allocation to Investment {i+1}, Generation {g+1}: {x[i][g].value()}")
        slt1.append(f'x[{i}][{g}]')
        slt2.append(x[i][g].value())
solution1 = pd.DataFrame(slt1)
solution2 = pd.DataFrame(slt2)
solution = pd.concat([solution1, solution2], axis=1)
solution.to_csv(r'/Users/kurozuhajime/Desktop/test.csv')

# 合計配分を確認
for i in range(num_investments):
    total_allocation = sum(x[i][g].value() for g in range(num_generations))
    print(f"Total Allocation for Investment {i+1}: {total_allocation}")

print(f"Total Budget Used: {sum(x[i][g].value() for i in range(num_investments) for g in range(num_generations))}")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/5v/b1rgw4cx4dsdgq25qzpwcwk40000gn/T/e6771fd9b0fe40f09282f43bebb74270-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/5v/b1rgw4cx4dsdgq25qzpwcwk40000gn/T/e6771fd9b0fe40f09282f43bebb74270-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 16 COLUMNS
At line 117 RHS
At line 129 BOUNDS
At line 130 ENDATA
Problem MODEL has 11 rows, 25 columns and 75 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 1 (-10) rows, 5 (-20) columns and 5 (-70) elements
0  Obj 225.5 Dual inf 19.799995 (5)
1  Obj 665.5
Optimal - objective value 665.5
After Postsolve, objective 665.5, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 665.5 - 1 iterations time 0.002, Presolve 0.00
Option for printingOptions change

In [4]:
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
import pandas as pd

# 問題のパラメータ
N = 5  # 投資項目数
N_i = ['health','education','economy','CO2','MF']
G = 5  # 世代数
G_i = ['baby','kids','youth','adult','elder']
S = 2  # 性別数
S_i = ['male','female']
I = 3  # 収入グループ数
I_i = ['poor','mediate','rich']

B = 100000  # 総予算
"""
R = [0.2, 0.25, 0.2, 0.15, 0.2]  # 各投資項目への最大割合
Q = [[[0.1, 0.1, 0.1], [0.15, 0.15, 0.1]],  # 世代1
     [[0.12, 0.12, 0.1], [0.13, 0.15, 0.1]],  # 世代2
     [[0.2, 0.15, 0.1], [0.1, 0.1, 0.1]],  # 世代3
     [[0.18, 0.2, 0.1], [0.1, 0.1, 0.1]],  # 世代4
     [[0.15, 0.15, 0.1], [0.1, 0.1, 0.1]]]  # 世代5

P = [[[[10, 12, 8], [8, 9, 7]],  # 投資項目1, 世代1
      [[15, 14, 13], [10, 11, 12]],  # 投資項目1, 世代2
      [[7, 6, 5], [5, 4, 3]],  # 投資項目1, 世代3
      [[12, 10, 8], [8, 7, 6]],  # 投資項目1, 世代4
      [[9, 8, 7], [6, 5, 4]]],  # 投資項目1, 世代5

     # 投資項目2
     [[[20, 18, 16], [15, 14, 12]],
      [[25, 24, 22], [22, 21, 20]],
      [[12, 13, 14], [10, 8, 9]],
      [[16, 15, 14], [14, 13, 12]],
      [[11, 10, 9], [8, 7, 6]]],

     # 投資項目3
     [[[5, 6, 7], [4, 3, 2]],
      [[10, 8, 6], [6, 7, 5]],
      [[3, 2, 1], [2, 1, 1]],
      [[8, 9, 7], [6, 5, 4]],
      [[6, 5, 4], [4, 3, 2]]],

     # 投資項目4
     [[[15, 14, 13], [12, 11, 10]],
      [[18, 17, 15], [14, 13, 12]],
      [[9, 8, 7], [7, 6, 5]],
      [[10, 11, 12], [8, 9, 10]],
      [[6, 7, 8], [5, 4, 3]]],

     # 投資項目5
     [[[7, 6, 5], [5, 4, 3]],
      [[10, 9, 8], [8, 7, 6]],
      [[3, 4, 5], [2, 3, 4]],
      [[12, 11, 10], [10, 9, 8]],
      [[9, 8, 7], [6, 5, 4]]]]


R = [0.2, 0.25, 0.2, 0.15, 0.2]  # 各投資項目への最大割合
Q = [[[0.1, 0.1, 0.1], [0.15, 0.15, 0.1]],  # 世代1
     [[0.12, 0.12, 0.1], [0.13, 0.15, 0.1]],  # 世代2
     [[0.2, 0.15, 0.1], [0.1, 0.1, 0.1]],  # 世代3
     [[0.18, 0.2, 0.1], [0.1, 0.1, 0.1]],  # 世代4
     [[0.15, 0.15, 0.1], [0.1, 0.1, 0.1]]]  # 世代5

# アウトカム寄与度（P）を重み付きで設定
# 高い重みを赤ちゃん、こども、若者（G=0,1,2）かつ 教育(n=1)、CO2(n=3)、マテリアルフットプリント(n=4) に設定
P = [[[[30 if g in [0, 1, 2] else 10 for i in range(I)] for s in range(S)] for g in range(G)] for n in range(N)]

# 特定の投資項目の寄与度を高く設定
for n in [1, 3, 4]:  # 教育, CO2削減, マテリアルフットプリント
    for g in range(G):
        for s in range(S):
            for i in range(I):
                if g in [0, 1, 2]:  # 赤ちゃん, こども, 若者
                    P[n][g][s][i] *= 1.5  # 重み付け（寄与度を1.5倍）

"""

R = [0.2, 0.25, 0.2, 0.15, 0.2]  # 各投資項目への最大割合
Q = [[[0.1, 0.1, 0.1], [0.15, 0.15, 0.1]],  # 世代1
     [[0.12, 0.12, 0.1], [0.13, 0.15, 0.1]],  # 世代2
     [[0.2, 0.15, 0.1], [0.1, 0.1, 0.1]],  # 世代3
     [[0.18, 0.2, 0.1], [0.1, 0.1, 0.1]],  # 世代4
     [[0.15, 0.15, 0.1], [0.1, 0.1, 0.1]]]  # 世代5

# アウトカム寄与度（P）を設定
P = [[[[10 for i in range(I)] for s in range(S)] for g in range(G)] for n in range(N)]

# 教育投資（n=1）に低収入・若年層の寄与度を高く設定
for g in [0, 1, 2]:  # 赤ちゃん、こども、若者
    for s in range(S):
        P[1][g][s][0] *= 5  # 低収入の寄与度を5倍

# 環境投資（n=3, 4）に低収入・高齢壮年層の寄与度を高く設定
for n in [3, 4]:  # CO2、マテリアルフットプリント
    for g in [3, 4]:  # 壮年、高齢者
        for s in range(S):
            P[n][g][s][0] *= 4  # 低収入の寄与度を4倍



# モデルの作成
model = LpProblem(name="investment-allocation", sense=LpMaximize)

# 変数の定義
x = [[[[
    LpVariable(f"x_{n}_{g}_{s}_{i}", lowBound=0)
    for i in range(I)]
    for s in range(S)]
    for g in range(G)]
    for n in range(N)]

# 目的関数の設定
model += lpSum(P[n][g][s][i] * x[n][g][s][i] for n in range(N) for g in range(G) for s in range(S) for i in range(I))

# 制約条件1: 総予算制約
model += lpSum(x[n][g][s][i] for n in range(N) for g in range(G) for s in range(S) for i in range(I)) <= B

# 制約条件2: 投資項目ごとの割合制約
for n in range(N):
    model += lpSum(x[n][g][s][i] for g in range(G) for s in range(S) for i in range(I)) <= R[n] * B

# 制約条件3: 世代・性別・収入別の割合制約
for g in range(G):
    for s in range(S):
        for i in range(I):
            model += lpSum(x[n][g][s][i] for n in range(N)) <= Q[g][s][i] * B

# 問題を解く
model.solve()

# 結果の出力
print("最適値:", model.objective.value())
slt1 = []
slt2 = []
slt3 = []
for n in range(N):
    for g in range(G):
        for s in range(S):
            for i in range(I):
                print(f"x_{n}_{g}_{s}_{i} = {x[n][g][s][i].value()}")
                slt1.append(f'x[{n}][{g}][{s}][{i}]')
                slt2.append(f'({N_i[n]})*({G_i[g]})*({S_i[s]})*({I_i[i]})')
                slt3.append(x[n][g][s][i].value())
solution1 = pd.DataFrame(slt1)
solution2 = pd.DataFrame(slt2)
solution3 = pd.DataFrame(slt3)
solution = pd.concat([solution1, solution2, solution3], axis=1)
solution.to_csv(r'/Users/kurozuhajime/Desktop/Pythonコード・データ/防災予算/防災データ/test.csv')

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/5v/b1rgw4cx4dsdgq25qzpwcwk40000gn/T/aff4329e002348a1a546194dfffe1e92-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/5v/b1rgw4cx4dsdgq25qzpwcwk40000gn/T/aff4329e002348a1a546194dfffe1e92-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 41 COLUMNS
At line 642 RHS
At line 679 BOUNDS
At line 680 ENDATA
Problem MODEL has 36 rows, 150 columns and 450 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 36 (0) rows, 150 (0) columns and 450 (0) elements
Perturbing problem by 0.001% of 50 - largest nonzero change 0.00013161687 ( 0.0013161687%) - largest zero change 0
0  Obj -0 Dual inf 1979.9893 (150)
0  Obj -0 Dual inf 1980 (150)
12  Obj 3050000
Optimal - objective value 3050000
Optimal objective 3050000